# DataCleanEnv — RL Training with GRPO

This notebook trains an LLM to clean messy data using GRPO (Group Relative Policy Optimization) from the `trl` library.

## Setup
1. Go to Runtime → Change runtime type → GPU (T4)
2. Run all cells
3. Monitor training progress

## Hardware Requirements
- **Minimum**: Colab T4 (16GB VRAM)
- **Recommended**: Colab A100 (40GB VRAM) for larger models
- **Alternative**: Your friend's RTX 4050 (6GB) with QLoRA on Qwen2.5-0.5B

## 1. Install Dependencies

In [ ]:
!pip install -q openenv-core trl transformers accelerate peft bitsandbytes wandb
!pip install -q pandas numpy fastapi fastmcp uvicorn

## 2. Configuration

In [ ]:
import os

# Model configuration
MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"  # Small model for 6GB VRAM
# For Colab T4 (16GB), you can use:
# MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
# MODEL_ID = "meta-llama/Llama-3.2-1B-Instruct"

# Training configuration
CONFIG = {
    "learning_rate": 1e-5,
    "num_train_epochs": 3,
    "per_device_train_batch_size": 1,
    "gradient_accumulation_steps": 4,
    "max_prompt_length": 1024,
    "max_completion_length": 2048,
    "num_generations": 4,  # Number of samples per prompt for GRPO
    "beta": 0.1,  # KL penalty
    "temperature": 0.7,
    "output_dir": "./dataclean-grpo-checkpoint",
    "logging_steps": 10,
    "save_steps": 50,
}

# HuggingFace token (for model push)
from huggingface_hub import login
# login()  # Uncomment to login interactively

## 3. Start Environment Server

In [ ]:
import subprocess
import time
import threading

# Start the FastAPI server in background
server_process = subprocess.Popen(
    ["uvicorn", "data_clean_env.server.app:app", "--host", "0.0.0.0", "--port", "8000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
)

# Wait for server to start
time.sleep(5)
print("✅ Environment server started on http://localhost:8000")

# Test health endpoint
import urllib.request
try:
    response = urllib.request.urlopen("http://localhost:8000/health")
    print(f"✅ Health check passed: {response.status}")
except Exception as e:
    print(f"⚠️ Server may not be ready yet: {e}")

## 4. Create Training Dataset

In [ ]:
from datasets import Dataset

# Training prompts for data cleaning tasks
prompts = [
    # Easy tasks
    "You are given a CSV file with customer data. Read the file, identify issues (duplicates, missing values), clean the data, and submit the cleaned file. The messy file is at: easy_messy.csv",
    "Clean this customer dataset. Remove duplicate rows and handle missing values appropriately. File: easy_messy.csv",
    # Medium tasks
    "Clean this sales records dataset with 20 rows. Issues include: inconsistent date formats, inconsistent casing, invalid emails, extra whitespace, wrong data types. File: medium_messy.csv",
    "Normalize the dates to YYYY-MM-DD, standardize casing to title case, validate emails, trim whitespace, and convert amounts to floats. File: medium_messy.csv",
    # Hard tasks
    "Clean this product inventory dataset with 50 rows. Fix: missing product IDs, negative quantities, invalid prices, inconsistent date formats, inconsistent category casing, extra whitespace, mixed price formats. File: hard_messy.csv",
    "Generate missing IDs, set negative quantities to 0, set invalid prices to 0, normalize dates, standardize casing, trim whitespace, extract numeric prices. File: hard_messy.csv",
]

# Create dataset
train_dataset = Dataset.from_dict({"prompt": prompts * 5})  # Repeat for more samples
print(f"✅ Created training dataset with {len(train_dataset)} samples")
print(f"Sample prompt: {train_dataset[0]['prompt'][:100]}...")

## 5. Define Reward Function

In [ ]:
import re
import json
import asyncio
from data_clean_env.client import DataCleanEnv
from data_clean_env.models import AVAILABLE_TOOLS
from openenv.core.env_server.mcp_types import CallToolAction

async def compute_reward_for_completion(prompt, completion):
    """
    Compute reward by executing the completion in the environment.
    
    The completion should contain tool calls that clean the data.
    We extract the submitted file path and grade it.
    """
    # Parse the completion to extract tool calls
    # For simplicity, we'll use a heuristic:
    # - Check if the completion contains cleaning operations
    # - Check if it submits a file
    
    has_read = "read_file" in completion
    has_python = "run_python" in completion or "import pandas" in completion
    has_write = "write_file" in completion
    has_submit = "submit_cleaned_file" in completion
    
    # Simple reward heuristic
    reward = 0.0
    if has_read:
        reward += 0.1
    if has_python:
        reward += 0.3
    if has_write:
        reward += 0.2
    if has_submit:
        reward += 0.4
    
    return reward

def reward_fn(completions, **kwargs):
    """
    Reward function for GRPO training.
    
    Args:
        completions: List of generated completions
    
    Returns:
        List of rewards (one per completion)
    """
    rewards = []
    for completion in completions:
        # Extract text from completion
        if isinstance(completion, list):
            text = " ".join([t["text"] for t in completion if "text" in t])
        else:
            text = str(completion)
        
        # Compute reward
        reward = 0.0
        if "read_file" in text:
            reward += 0.1
        if "run_python" in text or "import pandas" in text:
            reward += 0.3
        if "write_file" in text:
            reward += 0.2
        if "submit_cleaned_file" in text:
            reward += 0.4
        
        rewards.append(reward)
    
    return rewards

print("✅ Reward function defined")

## 6. Load Model and Tokenizer

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Load model with quantization for memory efficiency
from transformers import BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True,
)

print(f"✅ Model loaded: {MODEL_ID}")
print(f"✅ Parameters: {model.num_parameters() / 1e6:.1f}M")

## 7. Setup GRPO Trainer

In [ ]:
from trl import GRPOConfig, GRPOTrainer
from peft import LoraConfig

# LoRA configuration for memory-efficient training
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    task_type="CAUSAL_LM",
)

# GRPO training arguments
training_args = GRPOConfig(
    output_dir=CONFIG["output_dir"],
    learning_rate=CONFIG["learning_rate"],
    num_train_epochs=CONFIG["num_train_epochs"],
    per_device_train_batch_size=CONFIG["per_device_train_batch_size"],
    gradient_accumulation_steps=CONFIG["gradient_accumulation_steps"],
    max_prompt_length=CONFIG["max_prompt_length"],
    max_completion_length=CONFIG["max_completion_length"],
    num_generations=CONFIG["num_generations"],
    beta=CONFIG["beta"],
    temperature=CONFIG["temperature"],
    logging_steps=CONFIG["logging_steps"],
    save_steps=CONFIG["save_steps"],
    save_total_limit=2,
    report_to="none",  # Set to "wandb" if using Weights & Biases
    fp16=True,
    optim="adamw_8bit",
)

# Initialize trainer
trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=reward_fn,
    args=training_args,
    train_dataset=train_dataset,
    peft_config=lora_config,
)

print("✅ GRPO Trainer initialized")

## 8. Train the Model

In [ ]:
# Start training
print("🚀 Starting GRPO training...")
print(f"   Model: {MODEL_ID}")
print(f"   Training samples: {len(train_dataset)}")
print(f"   Epochs: {CONFIG['num_train_epochs']}")
print(f"   Batch size: {CONFIG['per_device_train_batch_size']}")
print(f"   Gradient accumulation: {CONFIG['gradient_accumulation_steps']}")
print(f"   Generations per prompt: {CONFIG['num_generations']}")

train_result = trainer.train()

# Save the model
trainer.save_model(CONFIG["output_dir"])
print(f"✅ Model saved to {CONFIG['output_dir']}")

## 9. Evaluate the Trained Model

In [ ]:
import asyncio
from openai import OpenAI

# Load the trained model for inference
from transformers import pipeline

# Create a text generation pipeline
pipe = pipeline(
    "text-generation",
    model=CONFIG["output_dir"],
    tokenizer=tokenizer,
    device_map="auto",
    torch_dtype=torch.float16,
)

# Test on a sample prompt
test_prompt = "Clean this customer dataset. Remove duplicate rows and handle missing values. File: easy_messy.csv"

messages = [
    {"role": "system", "content": "You are a data cleaning expert."},
    {"role": "user", "content": test_prompt},
]

output = pipe(
    messages,
    max_new_tokens=512,
    temperature=0.7,
    do_sample=True,
)

print("📝 Generated response:")
print(output[0]["generated_text"][-1]["content"])

## 10. Push to HuggingFace Hub (Optional)

In [ ]:
# Push the trained model to HuggingFace Hub
from huggingface_hub import login

# login()  # Uncomment and enter your token

# model.push_to_hub("your-username/dataclean-agent")
# tokenizer.push_to_hub("your-username/dataclean-agent")

# print("✅ Model pushed to HuggingFace Hub")

## 11. Cleanup

In [ ]:
# Stop the environment server
server_process.terminate()
server_process.wait()
print("✅ Environment server stopped")